# UniFi — train trên Kaggle

1. **Add Input** → thêm dataset Widar (thư mục chứa `userX/*.dat`).
2. **Settings** → Accelerator **GPU T4 x2** (hoặc P100), **Internet On**.
3. Sửa ô *Cấu hình* bên dưới → **Save Version → Save & Run All (Commit)** để chạy nền.
4. Kết quả nằm trong tab **Output**: `runs/<tên run>/` và `runs/<tên run>.zip`.

## Cấu hình

In [ ]:
REPO_URL  = 'https://github.com/phanvanhai/unifi.git'
BRANCH    = 'main'
CODE_DIR  = '/kaggle/working/unifi_code'  # nơi clone code (xoá cuối notebook để Output gọn)
DATA_ROOT = ''          # để trống → tự tìm trong /kaggle/input; hoặc ví dụ '/kaggle/input/widar3/widar'
OUTPUT    = '/kaggle/working'

# Tham số cho main.py (python main.py --help để xem hết)
# In-domain: chia ngẫu nhiên 80% train / 20% test theo từng lớp
ARGS = '--epochs 100 --split random --test-ratio 0.2'
# ARGS = '--max-samples 120 --epochs 2'          # chạy thử nhanh

# Repo private: tạo Kaggle Secret tên GITHUB_TOKEN (Add-ons → Secrets) rồi đặt True
PRIVATE_REPO = False

## Môi trường + dữ liệu

In [ ]:
!nvidia-smi
!nproc; free -g; df -h /kaggle/working /tmp | cat
!find /kaggle/input -maxdepth 4 -type d | head -40

In [ ]:
import os
from pathlib import Path

def find_widar(root='/kaggle/input', max_depth=6):
    """Thư mục cha chung của mọi thư mục userX/ có file .dat."""
    user_dirs = []
    for depth in range(1, max_depth + 1):
        for p in Path(root).glob('/'.join(['*'] * depth)):
            if p.is_dir() and p.name.startswith('user') and next(p.glob('*.dat'), None):
                user_dirs.append(str(p))
    return os.path.commonpath(user_dirs) if user_dirs else None

if not DATA_ROOT:
    DATA_ROOT = find_widar()
    assert DATA_ROOT, 'Không tìm thấy file .dat trong /kaggle/input — kiểm tra Add Input'
print('DATA_ROOT =', DATA_ROOT)
!ls {DATA_ROOT} | head -20

## Lấy code

In [ ]:
import os, subprocess
url = REPO_URL
if PRIVATE_REPO:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('GITHUB_TOKEN')
    url = REPO_URL.replace('https://', f'https://{token}@')
subprocess.run(['rm', '-rf', CODE_DIR], check=True)
subprocess.run(['git', 'clone', '--depth', '1', '-b', BRANCH, url, CODE_DIR], check=True)
!cd {CODE_DIR} && git log -1 --format='commit %h | %ci | %s' && ls

## Train

In [ ]:
!cd {CODE_DIR} && python -u main.py --data-root "{DATA_ROOT}" --output-dir "{OUTPUT}" {ARGS}

## Kết quả

In [ ]:
import glob, json
from IPython.display import Image, display
run = sorted(p for p in glob.glob(f'{OUTPUT}/runs/*') if os.path.isdir(p))[-1]
print('Run:', run)
print(json.dumps(json.load(open(f'{run}/summary.json')), indent=2, ensure_ascii=False))
print(open(f'{run}/plots/report.txt', encoding='utf-8').read())
for f in ['curves_loss', 'curves_acc', 'confusion_matrix', 'acc_per_view',
          'acc_by_user', 'acc_by_location', 'acc_by_orientation']:
    p = f'{run}/plots/{f}.png'
    if os.path.exists(p):
        display(Image(p))

In [ ]:
# Dọn code đã clone để tab Output chỉ còn runs/
!rm -rf {CODE_DIR}
!ls -la {OUTPUT}/runs